# Phase 2 — Prithvi-100M Feature Extraction

Stream HLS granules directly from NASA S3, run them through Prithvi-EO-1.0-100M, and save 768-dim feature vectors to S3.

**Run this notebook on an EC2 GPU instance** (`g4dn.xlarge` recommended). CPU is feasible for small tests but too slow for all 5 states × 20 years.


In [ ]:
import sys
sys.path.insert(0, "../src")

from hls_downloader import login, iter_granules_cloud
from privthi_extractor import load_prithvi, extract_features_from_array, extract_features_corn_only, save_features_to_s3
import torch

BUCKET = "cornsight-data"

# Use 'cuda' if on EC2 GPU, 'cpu' for local testing
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

login()
model = load_prithvi(device=DEVICE)


Using device: cpu
EarthData login successful.


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Prithvi loaded on cpu (PrithviViT, embed_dim=768)


## Quick test — 2 granules for Iowa 2023

Run this before the full extraction to confirm the pipeline works end-to-end.


In [3]:
for stacked, profile, gid in iter_granules_cloud("IA", 2023, "final", max_granules=2):
    features = extract_features_from_array(model, stacked, max_tiles=4)
    print(f"{gid}: features shape={features.shape}  mean={features.mean():.4f}")
    save_features_to_s3(features, gid, BUCKET, "IA", 2023, "final")
    print(f"  Saved to s3://{BUCKET}/processed/features/IA/2023/final/{gid}.npy")


/home/ec2-user/miniconda3/envs/Hackathon2026/lib/python3.11/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


  Found 4251 granules for IA 2023 [final]


/home/ec2-user/miniconda3/envs/Hackathon2026/lib/python3.11/site-packages/earthaccess/store.py:516: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/15 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/15 [00:00<?, ?it/s]

...

HLS.L30.T15TWH.2023121T164646.v2.0: features shape=(768,)  mean=-0.0008
  Saved to s3://cornsight-data/processed/features/IA/2023/final/HLS.L30.T15TWH.2023121T164646.v2.0.npy


QUEUEING TASKS | :   0%|          | 0/15 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/15 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/15 [00:00<?, ?it/s]

HLS.L30.T15TXH.2023121T164646.v2.0: features shape=(768,)  mean=-0.0026
  Saved to s3://cornsight-data/processed/features/IA/2023/final/HLS.L30.T15TXH.2023121T164646.v2.0.npy


## Full extraction — all 5 states × 2015–2024 × final forecast date

Run on EC2. Each state/year takes ~10–30 min on GPU depending on granule count.
Already-saved granules are skipped automatically.


In [ ]:
from privthi_extractor import list_features_in_s3, extract_features_corn_only

STATES = ["IA"]                # full prod: ["IA", "CO", "WI", "MO", "NE"]
YEARS = [2023]                 # head-to-head test vs existing `final/` for same year
GRANULE_DATE = "final"         # which HLS months to pull
SAVE_PREFIX  = "final-corn"    # s3 sub-prefix to write into (kept separate from `final/`)
MIN_CORN_FRAC = 0.05           # drop tiles with <5% corn pixels

for state in STATES:
    for year in YEARS:
        print(f"\n=== {state} {year} [{GRANULE_DATE} -> {SAVE_PREFIX}] ===")
        existing = list_features_in_s3(BUCKET, state, year, SAVE_PREFIX)
        print(f"  Already saved: {len(existing)} granules")

        count = 0; skipped_nocorn = 0
        for stacked, profile, gid in iter_granules_cloud(state, year, GRANULE_DATE):
            if any(gid in k for k in existing):
                continue
            try:
                feats, kept, total = extract_features_corn_only(
                    model, stacked, profile,
                    state_abbr=state, year=year, bucket=BUCKET,
                    min_corn_frac=MIN_CORN_FRAC, verbose=False,
                )
                if kept == 0:
                    skipped_nocorn += 1
                    continue
                save_features_to_s3(feats, gid, BUCKET, state, year, SAVE_PREFIX)
                count += 1
            except Exception as e:
                print(f"  WARN {gid}: {e}")

        print(f"  Newly saved: {count} granules  (skipped {skipped_nocorn} no-corn)")



=== IA 2023 [final -> final-corn] ===
  Already saved: 0 granules
  Found 4251 granules for IA 2023 [final]


QUEUEING TASKS | :   0%|          | 0/15 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/15 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/15 [00:00<?, ?it/s]

/home/ec2-user/miniconda3/envs/Hackathon2026/lib/python3.11/site-packages/earthaccess/store.py:516: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)


...

COLLECTING RESULTS | :   0%|          | 0/15 [00:00<?, ?it/s]

/home/ec2-user/miniconda3/envs/Hackathon2026/lib/python3.11/site-packages/earthaccess/store.py:516: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/15 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/15 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/15 [00:00<?, ?it/s]